In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

In [ ]:
base = Path.cwd().parent

In [ ]:
part1 = pd.read_json(base / "data" / "Streaming_History_Audio_2025_1.json")
part2 = pd.read_json(base / "data" / "Streaming_History_Audio_2022-2025_0.json")
df = pd.concat([part1, part2], ignore_index=True)

In [ ]:
df.dropna(axis=1, how='all', inplace=True)
df.drop(columns=['ip_addr'], inplace=True)

In [ ]:
df['ts'] = pd.to_datetime(df['ts'])
df['platform'] = df['platform'].astype('category')
df['conn_country'] = df['conn_country'].astype('category')
df['reason_start'] = df['reason_start'].astype('category')
df['reason_end'] = df['reason_end'].astype('category')
df['shuffle'] = df['shuffle'].astype('boolean')
df['skipped'] = df['skipped'].astype('boolean')
df['offline'] = df['offline'].astype('boolean')
df['incognito_mode'] = df['incognito_mode'].astype('boolean')

In [ ]:
df['ms_played'] = df['ms_played'] / 1000 / 60  # Convert to minutes

In [ ]:
df.info()

In [ ]:
df['ts'].hist(bins=100, figsize=(12, 6))

In [ ]:
tmp = df['master_metadata_album_artist_name'].value_counts()
plt.figure(figsize=(15, 8))
sns.barplot(x=tmp.index[:10], y=tmp.values[:10])
plt.xticks(rotation=45)
plt.title('Top 10 Artists by Number of Streams')
plt.xlabel('Artist')
plt.ylabel('Number of Streams')
# plt.tight_layout()
plt.show()

In [ ]:
df

In [ ]:
artist_stats = df.groupby('master_metadata_album_artist_name').agg(
    total_plays=('ts', 'count'),
    skips=('skipped', 'sum')
)
artist_stats['skip_rate'] = artist_stats['skips'] / artist_stats['total_plays']


In [ ]:
# only keep artists with a reasonable sample size, otherwise low-n noise dominates
filtered = artist_stats[artist_stats['total_plays'] >= 30].copy()

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(filtered['total_plays'], filtered['skip_rate'], alpha=0.6)

# label only the extremes so it's not cluttered
top_candidates = filtered.sort_values('skip_rate', ascending=False).head(5)
top_favorite = filtered.sort_values('total_plays', ascending=False).head(4)

for name, row in top_candidates.iterrows():
    ax.annotate(name, (row['total_plays'], row['skip_rate']), fontsize=8, xytext=(5,5), textcoords='offset points')
for name, row in top_favorite.iterrows():
    ax.annotate(name, (row['total_plays'], row['skip_rate']), fontsize=8, color='red', xytext=(5,5), textcoords='offset points')

ax.axhline(df['skipped'].mean(), color='gray', linestyle='--', label='your avg skip rate')
ax.set_xlabel('total_plays')
ax.set_ylabel('skip_rate')
ax.legend()
plt.show()

In [ ]:
track_stats = df.groupby('master_metadata_track_name').agg(
    total_plays=('ts', 'count'),
    skips=('skipped', 'sum')
)
track_stats['skip_rate'] = track_stats['skips'] / track_stats['total_plays']

# filter out artists you've barely played, otherwise n=1 100%-skip artists dominate
track_stats[track_stats['total_plays'] >= 10].sort_values('skip_rate', ascending=False)

In [ ]:
# only keep artists with a reasonable sample size, otherwise low-n noise dominates
filtered = track_stats[track_stats['total_plays'] >= 20].copy()

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(filtered['total_plays'], filtered['skip_rate'], alpha=0.6)

# label only the extremes so it's not cluttered
top_candidates = filtered.sort_values('skip_rate', ascending=False).head(5)
top_favorite = filtered.sort_values('total_plays', ascending=False).head(4)

for name, row in top_candidates.iterrows():
    ax.annotate(name, (row['total_plays'], row['skip_rate']), fontsize=8, xytext=(5,5), textcoords='offset points')
for name, row in top_favorite.iterrows():
    ax.annotate(name, (row['total_plays'], row['skip_rate']), fontsize=8, color='red', xytext=(5,5), textcoords='offset points')

ax.axhline(df['skipped'].mean(), color='gray', linestyle='--', label='your avg skip rate')
ax.set_xlabel('total_plays')
ax.set_ylabel('skip_rate')
ax.legend()
plt.show()